# Network Rail Incident Dataset Analysis

Loads the pre-processed Network Rail weather-related incident dataset and
produces summary statistics and visualisations used in the paper.

**Inputs**
- `all_data.pkl` — combined incident dataset (Network Rail records merged with
  Met Office weather-pattern classifications). This file is not publicly
  distributed; please contact the authors or refer to the data availability
  statement in the paper for access instructions.

**Outputs** (figures and tables — uncomment `savefig` / `to_csv` calls to save)
- Bar chart of incident type frequencies
- Bar chart of incident days per weather pattern (WP)
- Stacked bar chart of incident type proportions by WP
- Highest-impact days by incident count (99th-percentile threshold)
- Highest-impact days by accumulated PfPI minutes (99th-percentile threshold)
- Pickle files of high-impact day metadata used by `02_sequence_footprints_plotting.ipynb`

**Configuration:** Set `DATA_DIR` and `OUTPUT_DIR` in the cell below before running.


In [ ]:
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from pathlib import Path

# ── User configuration ──────────────────────────────────────────────────────
# Directory containing input data files (all_data.pkl, sorted_days_per_wp.pkl)
DATA_DIR = "data"

# Root directory for output files (figures, CSVs, pickles)
OUTPUT_DIR = "output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Percentile threshold used to define "high-impact" days
PERCENTILE_THRESHOLD = 99

# Year that the dataset begins (used for year-index calculations)
DATASET_START_YEAR = 1960

# ── Consistent colour scheme for incident types ──────────────────────────────
INCIDENT_TYPES = [
    'FLOODING', 'HIGH WIND', 'ICING', 'INF WEATHR',
    'KRS / BLKT', 'LIGHTNING', 'PNTS SNOW', 'SEV FLOOD', 'SEV SNOW',
    'FOG/SNOW', 'HEAT SPEED', 'MISC OBS', 'SPL WRKING', 'WEATHER'
]
INCIDENT_COLOURS = [
    'teal', 'dodgerblue', 'skyblue', 'slategray',
    'darkslategray', 'cyan', 'darkviolet', 'darkblue', 'purple',
    'mediumslateblue', 'brown', 'forestgreen', 'palegreen', 'tan'
]
COLOUR_MAP = dict(zip(INCIDENT_TYPES, INCIDENT_COLOURS))

# ── Weather-pattern group definitions ────────────────────────────────────────
# WP numbers belonging to each large-scale circulation regime
WP_GROUPS = {
    'Anticyclonic': [6, 9, 25, 27, 13, 12, 16, 17, 3, 18, 10],
    'Cyclonic':     [11, 28, 8, 20, 26, 30, 14, 24, 2, 21, 22, 7, 29],
    'Unbiased':     [19, 4, 23, 1, 15, 5],
}
WP_GROUP_COLOURS = {'Anticyclonic': '#68CFE0', 'Cyclonic': '#945794', 'Unbiased': '#FFB703'}

# Detailed ordering and grouping for the stacked-proportion plot
WP_REGIME_ORDER = [
    27, 19, 28, 25, 9, 6, 11,   # NAO-  (7)
    20, 26, 30, 23, 8, 4,       # NAO+  (6)
    14, 24, 13, 1,              # NW    (4)
    21, 15, 2, 12,              # SW    (4)
    22, 16, 5, 17,              # SH    (4)
    18, 3,                      # HP    (2)
    29, 7,                      # LC    (2)
    10,                         # A     (1)
]
REGIME_LABELS = ['ALL', 'NAO-', 'NAO+', 'NW', 'SW', 'SH', 'HP', 'LC', 'A']
REGIME_SIZES  = [1,      7,      6,      4,    4,    4,    2,    2,    1]

# Month abbreviation lookup (used when building footprint dictionaries)
MONTH_ABBREV = {
    1: 'jan', 2: 'feb', 3: 'mar', 4: 'apr', 5: 'may', 6: 'jun',
    7: 'jul', 8: 'aug', 9: 'sep', 10: 'oct', 11: 'nov', 12: 'dec'
}

print("Configuration complete.")


## 1. Load dataset

In [ ]:
all_data_df = pd.read_pickle(os.path.join(DATA_DIR, "all_data.pkl"))

# Remove non-weather asset heat incidents if present
all_data_df = all_data_df[all_data_df['Incident Reason Name'] != 'ASSET HEAT']

# Ensure PfPI Minutes is numeric
all_data_df['PfPI Minutes'] = pd.to_numeric(all_data_df['PfPI Minutes'], errors='coerce')

print(f"Dataset loaded: {len(all_data_df):,} rows, {all_data_df['Date'].nunique()} unique dates")
print(all_data_df.dtypes)


## 2. Incident type frequency

In [ ]:
incident_counts = all_data_df['Incident Reason Name'].value_counts().sort_values(ascending=False)
plot_colors = [COLOUR_MAP.get(inc, 'gray') for inc in incident_counts.index]

fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(x=incident_counts.index, y=incident_counts.values, palette=plot_colors, ax=ax)
ax.set_xlabel("Incident Reason", fontsize=14)
ax.set_ylabel("Number of Incidents", fontsize=14)
ax.tick_params(axis='x', rotation=45, labelsize=14)
ax.tick_params(axis='y', labelsize=14)
plt.tight_layout()
# fig.savefig(os.path.join(OUTPUT_DIR, 'incidents_hist.png'), dpi=300)
plt.show()

# Summary table
total_incidents = incident_counts.sum()
incident_table = pd.DataFrame({
    "Incident Type": incident_counts.index,
    "Count":         incident_counts.values,
    "Percentage":    (incident_counts.values / total_incidents * 100).round(2),
}).sort_values("Percentage", ascending=False).reset_index(drop=True)
print(incident_table)
# incident_table.to_csv(os.path.join(OUTPUT_DIR, "incident_type_counts.csv"), index=False)


## 3. Incident days per weather pattern (WP)

In [ ]:
# Aggregate: count unique incident days per WP
aggregated_df   = all_data_df.groupby(['Date', 'WP Number']).size().reset_index(name='IncidentCount')
days_per_wp     = aggregated_df.groupby('WP Number')['Date'].nunique().reset_index(name='NumberOfDays')
sorted_days_per_wp = days_per_wp.sort_values('NumberOfDays', ascending=False).copy()
sorted_days_per_wp['WP Number'] = sorted_days_per_wp['WP Number'].astype(int)

# Assign circulation-type groups
def classify_group(wp):
    for group, members in WP_GROUPS.items():
        if wp in members:
            return group
    return "Other"

sorted_days_per_wp['Group'] = sorted_days_per_wp['WP Number'].apply(classify_group)
bar_colors = [WP_GROUP_COLOURS[row['Group']] for _, row in sorted_days_per_wp.iterrows()]

fig, ax = plt.subplots(figsize=(12, 8))
ax.bar(sorted_days_per_wp['WP Number'].astype(str), sorted_days_per_wp['NumberOfDays'], color=bar_colors)
ax.set_xlabel('WP Number')
ax.set_ylabel('Number of Days')
ax.set_title('Number of Incident Days per Weather Pattern')
ax.tick_params(axis='x', rotation=45)

legend_patches = [mpatches.Patch(color=c, label=g) for g, c in WP_GROUP_COLOURS.items()]
ax.legend(handles=legend_patches, title='WP Types')
plt.tight_layout()
# fig.savefig(os.path.join(OUTPUT_DIR, 'incident_days_per_WP.png'), dpi=300)
plt.show()

group_summary = sorted_days_per_wp.groupby('Group')['NumberOfDays'].sum().reset_index()
print("\n=== Days per circulation group ===")
print(group_summary)
print("\n=== Detailed table (per WP) ===")
print(sorted_days_per_wp[['WP Number', 'NumberOfDays', 'Group']])

# sorted_days_per_wp.to_pickle(os.path.join(OUTPUT_DIR, 'sorted_days_per_wp.pkl'))


## 4. Incident type proportions by WP (stacked bar)

In [ ]:
# Build pivot: proportion of each incident type within each WP
incident_by_wp    = all_data_df.groupby(['WP Number', 'Incident Reason Name']).size().reset_index(name='Count')
incident_wp_pivot = incident_by_wp.pivot(index='WP Number', columns='Incident Reason Name', values='Count').fillna(0)
incident_wp_pivot.index = incident_wp_pivot.index.astype(int)
incident_wp_pivot = incident_wp_pivot[incident_wp_pivot.sum(axis=1) > 0]

total_incidents_per_wp = incident_wp_pivot.sum(axis=1)
incident_wp_scaled     = incident_wp_pivot.div(total_incidents_per_wp, axis=0)

# Overall proportions (for the 'ALL' bar)
total_incident_counts      = all_data_df['Incident Reason Name'].value_counts()
total_incident_proportions = total_incident_counts / total_incident_counts.sum()

# Combine ALL + per-WP rows, reindex to regime order
present_types     = incident_wp_pivot.columns.tolist()
custom_colour_map = {inc: COLOUR_MAP.get(inc, 'gray') for inc in present_types}

plot_df = incident_wp_scaled.copy()
plot_df.loc['ALL'] = total_incident_proportions.reindex(plot_df.columns).fillna(0)
valid_wps = [wp for wp in WP_REGIME_ORDER if wp in plot_df.index]
plot_df   = plot_df.reindex(['ALL'] + valid_wps)

total_incidents_plot = total_incidents_per_wp.copy()
total_incidents_plot.loc['ALL'] = int(total_incident_counts.sum())
total_incidents_plot = total_incidents_plot.reindex(['ALL'] + valid_wps)

# Compute group boundary positions for vertical dividers
group_sizes       = REGIME_SIZES
group_end_pos     = pd.Series(group_sizes).cumsum()
group_start_pos   = [0] + list(group_end_pos[:-1])
group_midpoints   = [(s + e - 1) / 2 for s, e in zip(group_start_pos, group_end_pos)]

bar_colors_stacked = [custom_colour_map[c] for c in plot_df.columns]

fig, ax = plt.subplots(figsize=(22, 10))
plot_df.plot(kind='bar', stacked=True, color=bar_colors_stacked, ax=ax, width=0.8)
ax.grid(False)

# Annotate totals above bars
for idx, total in enumerate(total_incidents_plot):
    if pd.notna(total):
        ax.text(idx, 1.02, f'{int(total)}', ha='center', va='bottom', fontsize=13)

# Group dividers
ax.axvline(x=0.5, color='black', linestyle='-', linewidth=1.2)
for pos in group_end_pos[1:-1]:
    ax.axvline(x=pos - 0.5, color='black', linestyle='--', linewidth=1.2)

ax.set_xticks(range(len(plot_df)))
ax.set_xticklabels(['ALL'] + valid_wps, rotation=0, fontsize=14)
ax.set_xlabel('Weather Pattern Number', fontsize=16)
ax.set_ylabel('Proportion of Incident Types', fontsize=16)
ax.tick_params(axis='y', labelsize=14)

# Top-axis regime labels
ax_top = ax.secondary_xaxis('top')
ax_top.set_xticks(group_midpoints)
ax_top.set_xticklabels(REGIME_LABELS, fontsize=14)
ax_top.spines['top'].set_visible(False)
ax_top.tick_params(axis='x', length=0)

legend_patches = [mpatches.Patch(color=c, label=inc) for inc, c in custom_colour_map.items()]
ax.legend(handles=legend_patches, title='Incident Types', title_fontsize=13, fontsize=12,
          bbox_to_anchor=(1, 1.01), loc='upper left')
plt.tight_layout()
# fig.savefig(os.path.join(OUTPUT_DIR, 'WP_incidents_proportions.png'), dpi=300)
plt.show()


## 5. High-impact days — by incident count (99th percentile)

In [ ]:
# ── Identify high-impact days by raw incident count ─────────────────────────
total_daily_incidents = all_data_df.groupby('Date').size().reset_index(name='Total Incidents')
threshold_count       = total_daily_incidents['Total Incidents'].quantile(PERCENTILE_THRESHOLD / 100)
filtered_days_count   = total_daily_incidents[total_daily_incidents['Total Incidents'] >= threshold_count]

filtered_data_count   = all_data_df[all_data_df['Date'].isin(filtered_days_count['Date'])]
incident_type_daily   = (filtered_data_count
                         .groupby(['Date', 'Incident Reason Name'])
                         .size().reset_index(name='Count'))
incident_type_daily   = pd.merge(incident_type_daily, filtered_days_count, on='Date')
incident_type_pivot_c = incident_type_daily.pivot(index='Date', columns='Incident Reason Name', values='Count').fillna(0)
incident_type_pivot_c['Total Incidents'] = incident_type_pivot_c.sum(axis=1)
incident_type_pivot_c = incident_type_pivot_c.sort_index()
stacked_count         = incident_type_pivot_c.drop(columns=['Total Incidents'])

# WP mapping for annotations
wp_mapping = all_data_df[['Date', 'WP Number']].drop_duplicates().set_index('Date')['WP Number']

incident_types_present = list(stacked_count.columns)
c_colour_map = {inc: COLOUR_MAP.get(inc, 'gray') for inc in incident_types_present}

fig, ax = plt.subplots(figsize=(14, 8))
stacked_count.plot(kind='bar', stacked=True,
                   color=[c_colour_map[inc] for inc in stacked_count.columns],
                   ax=ax, width=0.8)

for idx, date in enumerate(stacked_count.index):
    total_h  = stacked_count.loc[date].sum()
    wp_num   = wp_mapping.get(date, "N/A")
    inc_cnt  = int(incident_type_pivot_c['Total Incidents'].iloc[idx])
    ax.text(idx, total_h + 1,  f'WP: {wp_num}', ha='center', va='bottom', fontsize=12, weight='bold')
    ax.text(idx, total_h - 1,  f'{inc_cnt}',    ha='center', va='top',    fontsize=12, color='white', weight='bold')

ax.set_xlabel('Date', fontsize=13)
ax.set_ylabel('Incident Count', fontsize=13)
ax.set_xticks(range(len(stacked_count)))
ax.set_xticklabels(stacked_count.index.strftime('%Y-%m-%d'), rotation=45, fontsize=13)
plt.tight_layout()
# fig.savefig(os.path.join(OUTPUT_DIR, 'total_incidents_99P.png'), dpi=300)
plt.show()

# Headline statistic
pct = (filtered_days_count['Total Incidents'].sum() / total_daily_incidents['Total Incidents'].sum()) * 100
print(f"The top {100 - PERCENTILE_THRESHOLD}% of incident days account for {pct:.2f}% of all incidents.")


### 5a. Save high-impact-day metadata (incident count)

Saves a pickle file used by `02_sequence_footprints_plotting.ipynb` to look up
the meteorological footprint for each high-impact day.


In [ ]:
output_subdir = Path(OUTPUT_DIR) / "top_days_count_dictionaries"
output_subdir.mkdir(exist_ok=True)

footprint_dict_count = {}
for date in stacked_count.index:
    footprint_dict_count[date.strftime('%Y-%m-%d')] = {
        'input_month': MONTH_ABBREV[date.month],
        'year_index':  date.year - DATASET_START_YEAR,
        'day_index':   date.day - 1,   # zero-based
        'date':        date,
        'WP':          wp_mapping.get(date, None),
    }

out_path = output_subdir / "footprint_data.pkl"
with open(out_path, 'wb') as f:
    pickle.dump(footprint_dict_count, f)
print(f"Saved {len(footprint_dict_count)} entries to {out_path}")


## 6. High-impact days — by accumulated PfPI minutes (99th percentile)

In [ ]:
# ── Identify high-impact days by accumulated PfPI minutes ───────────────────
total_pfpi_daily    = all_data_df.groupby('Date')['PfPI Minutes'].sum().reset_index(name='Total PfPI Minutes')
threshold_pfpi      = total_pfpi_daily['Total PfPI Minutes'].quantile(PERCENTILE_THRESHOLD / 100)
filtered_days_pfpi  = total_pfpi_daily[total_pfpi_daily['Total PfPI Minutes'] >= threshold_pfpi]

filtered_data_pfpi  = all_data_df[all_data_df['Date'].isin(filtered_days_pfpi['Date'])]
incident_pfpi_daily = (filtered_data_pfpi
                       .groupby(['Date', 'Incident Reason Name'])['PfPI Minutes']
                       .sum().reset_index())
daily_inc_counts    = filtered_data_pfpi.groupby('Date').size().reset_index(name='Incident Count')
incident_pfpi_daily = pd.merge(incident_pfpi_daily, daily_inc_counts, on='Date')

incident_type_pivot_p = incident_pfpi_daily.pivot(index='Date', columns='Incident Reason Name',
                                                   values='PfPI Minutes').fillna(0)
incident_type_pivot_p['Total PfPI Minutes'] = incident_type_pivot_p.sum(axis=1)
incident_type_pivot_p = incident_type_pivot_p.sort_index()
incident_type_pivot_p = pd.merge(incident_type_pivot_p,
                                  daily_inc_counts.set_index('Date'),
                                  left_index=True, right_index=True)
stacked_pfpi = incident_type_pivot_p.drop(columns=['Total PfPI Minutes', 'Incident Count'])

pfpi_types_present = list(stacked_pfpi.columns)
p_colour_map = {inc: COLOUR_MAP.get(inc, 'gray') for inc in pfpi_types_present}

fig, ax = plt.subplots(figsize=(14, 8))
stacked_pfpi.plot(kind='bar', stacked=True,
                  color=[p_colour_map[inc] for inc in stacked_pfpi.columns],
                  ax=ax, width=0.8)

for idx, date in enumerate(stacked_pfpi.index):
    total_h  = incident_type_pivot_p['Total PfPI Minutes'].iloc[idx]
    inc_cnt  = incident_type_pivot_p['Incident Count'].iloc[idx]
    wp_num   = wp_mapping.get(date, "N/A")
    ax.text(idx, total_h + 20, f'WP: {wp_num}', ha='center', va='bottom', fontsize=12, weight='bold')
    ax.text(idx, total_h - 10, f'{inc_cnt}',     ha='center', va='top',    fontsize=12, color='white', weight='bold')

ax.set_xlabel('Date', fontsize=13)
ax.set_ylabel('Accumulated PfPI Minutes', fontsize=13)
ax.set_xticks(range(len(stacked_pfpi)))
ax.set_xticklabels(stacked_pfpi.index.strftime('%Y-%m-%d'), rotation=45, fontsize=10)
plt.tight_layout()
# fig.savefig(os.path.join(OUTPUT_DIR, 'PfPI_days_99P.png'), dpi=300)
plt.show()

pct_pfpi = (filtered_days_pfpi['Total PfPI Minutes'].sum() / total_pfpi_daily['Total PfPI Minutes'].sum()) * 100
print(f"The top {100 - PERCENTILE_THRESHOLD}% of PfPI days account for {pct_pfpi:.2f}% of total PfPI Minutes.")


### 6a. Save high-impact-day metadata (PfPI)


In [ ]:
output_subdir_pfpi = Path(OUTPUT_DIR) / "top_days_pfpi_dictionaries"
output_subdir_pfpi.mkdir(exist_ok=True)

footprint_dict_pfpi = {}
for date in stacked_pfpi.index:
    footprint_dict_pfpi[date.strftime('%Y-%m-%d')] = {
        'input_month': MONTH_ABBREV[date.month],
        'year_index':  date.year - DATASET_START_YEAR,
        'day_index':   date.day - 1,
        'date':        date,
        'WP':          wp_mapping.get(date, None),
    }

out_path_pfpi = output_subdir_pfpi / "footprint_data.pkl"
with open(out_path_pfpi, 'wb') as f:
    pickle.dump(footprint_dict_pfpi, f)
print(f"Saved {len(footprint_dict_pfpi)} entries to {out_path_pfpi}")


## 7. Exploratory queries

In [ ]:
# ── Search for incidents on a specific date ──────────────────────────────────
specific_date = "2012-01-03"   # edit as needed
pd.set_option('display.max_rows', None)
print(all_data_df[all_data_df['Date'] == specific_date])
pd.reset_option('display.max_rows')


In [ ]:
# ── Top 10 highest-incident days ─────────────────────────────────────────────
daily_incidents = all_data_df.groupby(['Date', 'WP Number']).size().reset_index(name='Incident Count')
print(daily_incidents.sort_values('Incident Count', ascending=False).head(10))


In [ ]:
# ── Top 50 longest individual incidents ──────────────────────────────────────
print(all_data_df.nlargest(50, 'Incident Duration'))


In [ ]:
# ── Daily PfPI summary ────────────────────────────────────────────────────────
daily_pfpi = all_data_df.groupby('Date').agg(
    PfPI_Minutes=('PfPI Minutes', 'sum'),
    Number_of_Incidents=('WP Number', 'count')
).reset_index().sort_values('PfPI_Minutes', ascending=False)
print(daily_pfpi.head(20))
